# 01 · Preprocessing

QC → normalize → HVG → embedding (regress · PCA · Harmony · UMAP · Leiden) → save

**In** — raw `.h5ad`
**Out** — `{dataset}_preprocessed.h5ad`
- `.X` = log-normalized, all genes
- `layers['counts']` = raw counts, all genes
- `obsm` = `X_pca`, `X_pca_harmony`, `X_umap`

**Depth is handled in two independent places and they do not overlap.** Here,
`regress_out` removes depth from the *embedding* so clustering is not driven by
sequencing depth. The saved expression is rebuilt from `.raw` and re-normalized,
so it carries no regression. Module 02 applies Pearson residuals to that saved
expression, and that is the only depth correction reaching SenePy.

Run in order from a fresh kernel.

## Config

In [ ]:
# ===========================================================================
# CONFIG - the only cell you edit
# ===========================================================================
import os
from pathlib import Path

DATASET = 'psychad_aging'    # psychad_aging | psychad_ad | psychencode | mathys

# Data root. Set once in your shell:
#   export SENESCENCE_DATA=/path/to/senescence_analysis
DATA_ROOT = Path(os.environ.get('SENESCENCE_DATA', 'data'))

DATASET_CONFIG = {
    'psychad_aging': {'input_file': 'raw/psychad/syn2580853_aging.h5ad',
                      'batch_key': ['Sample', 'Cohort'],    'cell_type_column': 'subclass'},
    'psychad_ad':    {'input_file': 'raw/psychad/syn2580853_cases.h5ad',
                      'batch_key': ['Sample', 'Cohort'],    'cell_type_column': 'subclass'},
    'psychencode':   {'input_file': 'raw/psychencode/PsychENCODE_merged.h5ad',
                      'batch_key': ['sample_id', 'Cohort'], 'cell_type_column': 'major_celltype'},
    'mathys':        {'input_file': None,        # TODO: path not yet recorded
                      'batch_key': None, 'cell_type_column': None},
}

_c               = DATASET_CONFIG[DATASET]
INPUT_FILE       = DATA_ROOT / _c['input_file']
BATCH_KEY        = _c['batch_key']
CELL_TYPE_COLUMN = _c['cell_type_column']

# -- QC thresholds. Set any to None to disable that filter deliberately. -----
MIN_GENES, MAX_GENES = 200, 40_000     # empty droplets / likely doublets
MIN_COUNTS           = 300
MAX_MT_PCT           = 5.0             # dying cells
MAX_RIBO_PCT         = 50.0            # degraded cells
MAX_HGB_PCT          = 10.0            # RBC contamination
MIN_CELLS            = 3               # gene seen in >= this many cells

# -- processing -------------------------------------------------------------
CONVERT_GENE_IDS  = False              # True if var_names are ENSG not symbols
TARGET_SUM        = 10_000
N_TOP_GENES       = 2_000
N_PCS             = 50
N_NEIGHBORS       = 30
LEIDEN_RESOLUTION = 0.8
REGRESS_VARS      = ['total_counts', 'pct_counts_mt']   # embedding only
SEED              = 0                  # matches scanpy defaults

# -- outputs ----------------------------------------------------------------
OUTPUT_FILE = DATA_ROOT / 'processed' / f'{DATASET}_preprocessed.h5ad'
FIGURES_DIR = DATA_ROOT / 'figures' / '01_preprocessing' / DATASET
RESULTS_DIR = DATA_ROOT / 'results' / '01_preprocessing' / DATASET
for d in (OUTPUT_FILE.parent, FIGURES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"  dataset    : {DATASET}")
print(f"  input      : {INPUT_FILE}")
print(f"  batch key  : {BATCH_KEY}")
print(f"  cell types : {CELL_TYPE_COLUMN}")
print(f"  output     : {OUTPUT_FILE}")
if not INPUT_FILE.exists():
    print("\n  WARNING input not found - check $SENESCENCE_DATA")

## Setup

In [ ]:
import random, platform
import scanpy as sc
import scanpy.external as sce          # harmony_integrate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import warnings; warnings.filterwarnings('ignore')

random.seed(SEED); np.random.seed(SEED)
sc.settings.verbosity = 1
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10,
    'axes.labelsize': 10, 'axes.titlesize': 11, 'legend.fontsize': 9,
    'font.family': 'sans-serif', 'axes.linewidth': 1.0,
    'axes.grid': False, 'pdf.fonttype': 42,
})
sc.settings.set_figure_params(dpi=150, dpi_save=300, facecolor='white', frameon=False)

# run record on disk, so a figure can be traced to the run that produced it
(RESULTS_DIR / 'run_info.txt').write_text(
    f"module  : 01_preprocessing\ndataset : {DATASET}\ninput   : {INPUT_FILE}\n"
    f"seed    : {SEED}\npython  : {platform.python_version()}\n"
    f"scanpy  : {sc.__version__}\nnumpy   : {np.__version__}\npandas  : {pd.__version__}\n")

print(f"scanpy {sc.__version__} - numpy {np.__version__} - pandas {pd.__version__}")

## Load

**Why.** Batch key and cell-type column differ by cohort (`Sample`/`Cohort` vs
`sample_id`/`Cohort`; `subclass` vs `major_celltype`). Failing here beats failing
inside Harmony forty cells later.

In [ ]:
adata = sc.read_h5ad(INPUT_FILE)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  genes look like: {list(adata.var_names[:4])}")

for key in (BATCH_KEY if isinstance(BATCH_KEY, list) else [BATCH_KEY]):
    if key not in adata.obs.columns:
        raise ValueError(f"batch key '{key}' not in obs - have: {list(adata.obs.columns)}")
    print(f"  batch '{key}': {adata.obs[key].nunique()} levels")

if CELL_TYPE_COLUMN not in adata.obs.columns:
    raise ValueError(f"cell type column '{CELL_TYPE_COLUMN}' not in obs")
print(f"  cell types '{CELL_TYPE_COLUMN}': {adata.obs[CELL_TYPE_COLUMN].nunique()}")

## Cell-type harmonization

**Why.** Cohorts use different vocabularies for the same populations - `Exc`/`EN_*`
vs `Excitatory`, `Oligo`/`ODC` vs `Oligodendrocyte`. Mapping to one vocabulary is
what makes a cell type mean the same thing across cohorts; without it, pooled
estimates compare differently-defined populations. Original labels are kept in
`{col}_original` so the mapping is auditable.

In [ ]:
HARMONIZATION_MAP = {
    "Exc": "Excitatory", "Ex": "Excitatory", "Excitatory neurons": "Excitatory",
    "Inh": "Inhibitory", "In": "Inhibitory", "Inhibitory neurons": "Inhibitory",
    "Astro": "Astrocyte", "Astrocytes": "Astrocyte", "AST": "Astrocyte",
    "Oligo": "Oligodendrocyte", "Olig": "Oligodendrocyte",
    "ODC": "Oligodendrocyte", "Oligodendrocytes": "Oligodendrocyte",
    "Oligodendrocyte precursor": "OPC", "opc": "OPC", "OPCs": "OPC",
    "Micro": "Microglia", "MG": "Microglia", "Mg": "Microglia",
    "Endo": "Endothelial", "EC": "Endothelial",
    "PC": "Pericyte", "Peri": "Pericyte", "SMC": "VSMC",
    "Vascular": "Vascular", "Immune": "Immune",
}

adata.obs[f"{CELL_TYPE_COLUMN}_original"] = adata.obs[CELL_TYPE_COLUMN].copy()
adata.obs[CELL_TYPE_COLUMN] = adata.obs[CELL_TYPE_COLUMN].astype(str)
n_before = adata.obs[CELL_TYPE_COLUMN].nunique()

changes = []
for orig, std in HARMONIZATION_MAP.items():
    m = adata.obs[CELL_TYPE_COLUMN] == orig
    if m.any():
        adata.obs.loc[m, CELL_TYPE_COLUMN] = std
        changes.append(f"    {orig} -> {std}  ({m.sum():,})")

for prefix, std in (('EN_', 'Excitatory'), ('IN_', 'Inhibitory')):
    m = adata.obs[CELL_TYPE_COLUMN].str.startswith(prefix)
    if m.any():
        adata.obs.loc[m, CELL_TYPE_COLUMN] = std
        changes.append(f"    {prefix}* -> {std}  ({m.sum():,})")

print("\n".join(changes) if changes else "    no changes needed")
print(f"\n  {n_before} -> {adata.obs[CELL_TYPE_COLUMN].nunique()} cell types")
print(adata.obs[CELL_TYPE_COLUMN].value_counts().to_string())

## Gene identifiers

**Why.** Everything downstream keys on gene symbols - QC masks (`MT-`, `RP[SL]`,
`HB[AB]`), SenePy hubs, hallmark panels. If `var_names` are Ensembl IDs the QC
masks match nothing and the MT filter silently passes every cell.

Conversion is many-to-one, so duplicate symbols are suffixed rather than dropped.

In [ ]:
_looks_ensembl = any(g.startswith('ENS') for g in adata.var_names[:10])
print(f"  var_names look like : {list(adata.var_names[:4])}")
print(f"  Ensembl detected    : {_looks_ensembl}")

if CONVERT_GENE_IDS and _looks_ensembl:
    import mygene
    mg = mygene.MyGeneInfo()

    mapping, unconverted = {}, []
    genes = adata.var_names.tolist()
    for i in range(0, len(genes), 1000):
        for r in mg.querymany(genes[i:i+1000], scopes='ensembl.gene',
                              fields='symbol', species='human', verbose=False):
            if 'symbol' in r:
                mapping[r['query']] = r['symbol']
            else:
                mapping[r['query']] = r['query']; unconverted.append(r['query'])

    new = pd.Series([mapping.get(g, g) for g in adata.var_names])
    dup = new.groupby(new).cumcount()                  # symbol collisions -> suffix
    new.loc[dup > 0] += '-' + dup.loc[dup > 0].astype(str)
    adata.var_names = pd.Index(new.values, name='gene_symbol')

    print(f"  converted           : {len(genes)-len(unconverted):,} / {len(genes):,}")
    print(f"  kept as Ensembl     : {len(unconverted):,}")
    print(f"  duplicates suffixed : {int((dup > 0).sum()):,}")

elif _looks_ensembl:
    print("\n  WARNING Ensembl IDs detected but CONVERT_GENE_IDS = False.")
    print("    QC masks and all gene panels downstream will fail to match.")
else:
    print("  already symbols - no conversion needed")

## QC metrics

**Why.** Mitochondrial fraction flags dying cells, ribosomal fraction flags
degraded ones, haemoglobin flags red-cell contamination. Computed before any
filtering so thresholds are chosen from the distributions rather than assumed.

The guard exists because an empty mask makes the corresponding filter a silent
no-op: `pct_counts_mt` would be 0 for every cell and the MT filter would pass
100% of cells while appearing to work.

In [ ]:
adata.var['mt']   = adata.var_names.str.upper().str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.upper().str.match('^RP[SL]')
adata.var['hgb']  = adata.var_names.str.upper().str.contains('^HB[AB]')

n_mt, n_ribo, n_hgb = (int(adata.var[k].sum()) for k in ('mt', 'ribo', 'hgb'))
print(f"  MT genes   : {n_mt}")
print(f"  ribo genes : {n_ribo}")
print(f"  hgb genes  : {n_hgb}")

if n_mt == 0 and MAX_MT_PCT is not None:
    raise ValueError(
        "No MT- genes matched var_names.\n"
        "  pct_counts_mt would be 0 for every cell and the MT filter would pass\n"
        "  100% of cells while appearing to work.\n"
        "  Either var_names are not gene symbols (set CONVERT_GENE_IDS = True),\n"
        "  or this reference excludes the mitochondrial genome - in which case\n"
        "  set MAX_MT_PCT = None to disable the filter deliberately.")
if n_ribo == 0 and MAX_RIBO_PCT is not None:
    print("  WARNING no ribosomal genes matched - ribo filter will be a no-op")
if n_hgb == 0 and MAX_HGB_PCT is not None:
    print("  WARNING no haemoglobin genes matched - HGB filter will be a no-op")

sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo', 'hgb'], inplace=True)

print(f"\n  median genes/cell  : {adata.obs['n_genes_by_counts'].median():,.0f}")
print(f"  median counts/cell : {adata.obs['total_counts'].median():,.0f}")
print(f"  median MT%         : {adata.obs['pct_counts_mt'].median():.2f}")
print(f"  median ribo%       : {adata.obs['pct_counts_ribo'].median():.2f}")
print(f"  median HGB%        : {adata.obs['pct_counts_hgb'].median():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (col, thr, lab) in zip(axes, [
        ('n_genes_by_counts', MIN_GENES,    'genes / cell'),
        ('total_counts',      MIN_COUNTS,   'UMI / cell'),
        ('pct_counts_mt',     MAX_MT_PCT,   'MT %'),
        ('pct_counts_ribo',   MAX_RIBO_PCT, 'ribo %')]):
    ax.hist(adata.obs[col], bins=100, color='#4E79A7')
    if thr is not None:
        ax.axvline(thr, color='#E15759', ls='--', lw=1)
    ax.set_xlabel(lab); ax.set_yticks([])
    if col in ('n_genes_by_counts', 'total_counts'): ax.set_xscale('log')
    for s in ('top', 'right', 'left'): ax.spines[s].set_visible(False)
fig.suptitle(f'{DATASET} - QC distributions (dashed = threshold)', y=1.04)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'qc_distributions.pdf', bbox_inches='tight')
plt.show()

## QC filtering

**Why.** Counting failures per threshold separately shows which filter is doing
the work. A single combined count hides the case where one threshold removes
almost everything - which is the failure mode worth catching before, not after.

Thresholds set to `None` in the config are skipped rather than compared against.

In [ ]:
n0, g0 = adata.n_obs, adata.n_vars

checks = {}
if MIN_GENES    is not None: checks[f'genes < {MIN_GENES}']    = adata.obs['n_genes_by_counts'] < MIN_GENES
if MAX_GENES    is not None: checks[f'genes > {MAX_GENES:,}']  = adata.obs['n_genes_by_counts'] > MAX_GENES
if MIN_COUNTS   is not None: checks[f'counts < {MIN_COUNTS}']  = adata.obs['total_counts'] < MIN_COUNTS
if MAX_MT_PCT   is not None: checks[f'MT% > {MAX_MT_PCT}']     = adata.obs['pct_counts_mt'] > MAX_MT_PCT
if MAX_RIBO_PCT is not None: checks[f'ribo% > {MAX_RIBO_PCT}'] = adata.obs['pct_counts_ribo'] > MAX_RIBO_PCT
if MAX_HGB_PCT  is not None: checks[f'HGB% > {MAX_HGB_PCT}']   = adata.obs['pct_counts_hgb'] > MAX_HGB_PCT

for lab, mask in checks.items():
    print(f"    {lab:<22} {mask.sum():>8,}  ({100*mask.mean():.1f}%)")

keep = ~np.logical_or.reduce([m.values for m in checks.values()])
adata = adata[keep, :].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)

print(f"\n  cells {n0:,} -> {adata.n_obs:,}  ({100*(n0-adata.n_obs)/n0:.1f}% removed)")
print(f"  genes {g0:,} -> {adata.n_vars:,}  ({100*(g0-adata.n_vars)/g0:.1f}% removed)")

## Normalize

**Why.** Raw counts are stashed to `layers['counts']` *before* normalizing - they
are needed for HVG selection (`seurat_v3` operates on counts), for the `.raw`
snapshot used at save, and by the DEG module for pseudobulk. Normalizing without
stashing first destroys them irrecoverably.

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)
print(f"  layers['counts'] : raw, {adata.layers['counts'].shape}")
print(f"  .X               : log-normalized to {TARGET_SUM:,}/cell")

## Highly variable genes

**Why.** `seurat_v3` selects on raw counts, not the normalized matrix - hence
`layer='counts'`. `subset=False` keeps all genes in the object; the HVG mask is
applied only to the embedding path below.

`adata.raw` is snapshotted here holding **all genes with raw counts**. That
snapshot is what the save cell restores from, and it is why the depth regression
below never reaches the saved expression.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES,
                            flavor='seurat_v3', layer='counts', subset=False)
print(f"  {adata.var['highly_variable'].sum():,} HVGs selected")

_snap   = adata.copy()
_snap.X = adata.layers['counts'].copy()
adata.raw = _snap
del _snap
print(f"  .raw snapshot    : {adata.raw.n_obs:,} x {adata.raw.n_vars:,} (raw counts, all genes)")

## Embedding - regress, scale, PCA

**Why.** Depth is regressed out here so clusters reflect biology rather than how
deeply a cell was sequenced. This affects the embedding **only** - the saved
expression is rebuilt from `.raw` below.

**Reordered vs source:** HVG subsetting now happens *before* `regress_out` rather
than after. Only the HVGs feed PCA, so the embedding is identical, but regressing
2,000 genes instead of ~30,000 is dramatically cheaper on a 500K-cell object.
Swap the two statements to restore source order.

In [ ]:
adata = adata[:, adata.var['highly_variable']].copy()      # <- moved ahead of regress
print(f"  subset to HVGs   : {adata.n_obs:,} x {adata.n_vars:,}")

sc.pp.regress_out(adata, REGRESS_VARS)
print(f"  regressed out    : {REGRESS_VARS}  (embedding only)")

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')
print(f"  PCA              : {N_PCS} PCs, "
      f"{adata.uns['pca']['variance_ratio'][:N_PCS].sum():.1%} variance explained")

In [ ]:
_keys = BATCH_KEY if isinstance(BATCH_KEY, list) else [BATCH_KEY]
for k in _keys:                                     # source printed a Series here
    print(f"  batch '{k}': {adata.obs[k].nunique()} levels")

sce.pp.harmony_integrate(adata, key=BATCH_KEY, basis='X_pca',
                         max_iter_harmony=20, adjusted_basis='X_pca_harmony')
print(f"  obsm['X_pca_harmony'] : {adata.obsm['X_pca_harmony'].shape}")

In [ ]:
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS,
                use_rep='X_pca_harmony', random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, random_state=SEED)
print(f"  leiden clusters  : {adata.obs['leiden'].nunique()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sc.pl.umap(adata, color=_keys[-1],        ax=axes[0], show=False, frameon=False,
           title='batch (post-Harmony)', legend_loc=None, s=2)
sc.pl.umap(adata, color=CELL_TYPE_COLUMN, ax=axes[1], show=False, frameon=False,
           title='cell type', legend_fontsize=6, s=2)
sc.pl.umap(adata, color='leiden',         ax=axes[2], show=False, frameon=False,
           title=f'leiden (res={LEIDEN_RESOLUTION})', legend_fontsize=6, s=2)
fig.suptitle(f'{DATASET} - embedding', y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'umap_overview.pdf', bbox_inches='tight')
plt.show()

## Save

**Why.** The working object is now HVG-only, scaled and regressed - unusable as
expression. The saved object is rebuilt from the `.raw` snapshot: all genes, raw
counts to `layers['counts']`, `.X` re-normalized from scratch. Embeddings and
cluster labels transfer across.

This is the step that keeps the depth regression out of the saved expression.

In [ ]:
full = adata.raw.to_adata()                       # all genes, raw counts in .X
full.layers['counts'] = full.X.copy()

full.uns.pop('log1p', None)
sc.pp.normalize_total(full, target_sum=TARGET_SUM)
sc.pp.log1p(full)

_log1p = full.uns.get('log1p')                    # transferring uns would clobber this
full.obsm, full.obsp, full.obs = adata.obsm.copy(), adata.obsp.copy(), adata.obs.copy()
full.uns = adata.uns.copy()
if _log1p is not None:
    full.uns['log1p'] = _log1p

full.var['highly_variable'] = False
full.var.loc[adata.var_names, 'highly_variable'] = True

full.write_h5ad(OUTPUT_FILE)
adata = full
print(f"  saved {OUTPUT_FILE.name}  ({OUTPUT_FILE.stat().st_size/1e9:.2f} GB)")
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes - "
      f"{adata.var['highly_variable'].sum():,} HVGs flagged")

## Gate

**Why.** Module 02 requires both `.X` log-normalized and `layers['counts']` raw.
Checking here costs seconds; discovering it in module 02 costs a reload.

In [ ]:
def _gate(label, ok, detail=''):
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}{'  - ' + detail if detail else ''}")
    if not ok:
        raise AssertionError(f"GATE FAILED: {label}. {detail}")
    return ok

_gate("layers['counts'] present", 'counts' in adata.layers)
_c = adata.layers['counts']
_s = _c.data[:1000] if sp.issparse(_c) else np.asarray(_c).flat[:1000]
_gate("counts are integers", bool(np.allclose(_s, np.round(_s))))
_gate(".X is log-normalized (non-negative)",
      float(adata.X.data.min() if sp.issparse(adata.X) else adata.X.min()) >= 0)
_gate("cell-type column present", CELL_TYPE_COLUMN in adata.obs, CELL_TYPE_COLUMN)
_gate("no all-zero cells", int((np.asarray(_c.sum(axis=1)).ravel() == 0).sum()) == 0)
_gate("embeddings attached", 'X_pca_harmony' in adata.obsm and 'X_umap' in adata.obsm)

print(f"\n  -> ready for module 02 - {adata.n_obs:,} x {adata.n_vars:,}")